# Link Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Define the path to the file inside the shared folder
os.chdir('/content/drive/My Drive/DBEC/model')

!ls

In [ ]:
pip install pyreadr

In [ ]:
from run_pipeline import *
import numpy as np
import pandas as pd
import torch
import pyreadr
import time
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from rpy2.robjects import globalenv
from rpy2.robjects import BoolVector
import rpy2.robjects as robjects
from rpy2.robjects import numpy2ri
from rpy2.robjects.packages import importr

# Activate NumPy-to-R conversion globally
numpy2ri.activate()
from rpy2.robjects import pandas2ri
pandas2ri.activate()


def run_seed(seed):
  # Set seed for PyTorch
  seed = 42
  torch.manual_seed(seed)
  # If using CUDA (GPU), set the seed for all GPUs
  torch.cuda.manual_seed_all(seed)
  # Set the seed for NumPy, if you are using it in conjunction with PyTorch
  np.random.seed(seed)


# device = torch.device("cuda")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Experiment on {device}...")

# load R functions

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install.packages('SuperLearner')
install.packages('earth')
install.packages('glmnet')
install.packages('ranger')
install.packages('e1071')
install.packages('BalancedSampling')
install.packages('MatchIt')
install.packages('ebal')
install.packages('WeightIt')
install.packages('osqp')
install.packages('kbal')
install.packages('caret')
install.packages('sampling')


library(SuperLearner)
library(earth)
library(glmnet)
library(ranger)
library(e1071)
library(BalancedSampling)
library(MatchIt)
library(ebal)
library(parallel)
library(kbal)
library(WeightIt)
library(osqp)
library(caret)
library(sampling)

In [ ]:
%%R

####  Function indPE calculates the propensity scores index
# a library of machine learning algorithms
# (penalized regression, random forests, and multivariate adaptive regression splines)
sl_libs <- c('SL.glmnet', 'SL.ranger', 'SL.earth', 'SL.glm', 'SL.gam', 'SL.nnet')

indPE <- function(X, Y, sl_libs, nrep=30) {
  cores <- max(1, detectCores()-1)
  cl <- makeCluster(cores); on.exit(stopCluster(cl), add=TRUE)

  # 1) load pkgs on workers
  clusterEvalQ(cl, {
    suppressPackageStartupMessages({
      library(SuperLearner); library(glmnet); library(ranger); library(earth); library(ranger); library(e1071)
    })
    NULL
  })

  clusterExport(cl, c("X","Y","sl_libs"), envir=environment())

  rr <- parSapply(cl, 1:nrep, function(i){
      fit <- SuperLearner(Y=Y, X=X, family=binomial(), SL.library=sl_libs)
    sd(as.vector(predict(fit)$pred))
  })

  mean(rr, na.rm=TRUE)
}


sl_score = function(trt, crl){

    colnames(crl) = colnames(trt)
    x_clinic = rbind(trt, crl)
    Trt = c(rep(1, nrow(trt)), rep(0, nrow(crl)))
    prop_score_sd = indPE(X = x_clinic, Y = Trt, sl_libs, nrep = 30)
    print(prop_score_sd)

    return(prop_score_sd)
}

# Define the min-max scaling function
minmax_scale <- function(col) {
  (col - min(col, na.rm = TRUE)) / (max(col, na.rm = TRUE) - min(col, na.rm = TRUE))
}



do_lpm = function(w_opt, x_pc_id, use_trt, use_pc, pro_pc, w_adjust){
    print(w_adjust)
    n_trt = nrow(use_trt)
    w_opt_vec = unlist(w_opt)
    w_re = w_opt_vec * n_trt / sum(w_opt_vec)

    # This automatically caps at 1 and redistributes the rest
    if(w_adjust=="incl"){
        w_lpm = inclusionprobabilities(w_re, n_trt)
    }else if(w_adjust=="pento1"){
        w_lpm = w_opt_vec
    }else{
        w_lpm = w_re
    }
    crl_lpm_ind = lpm1(prob=w_lpm, x=as.matrix(use_pc), type="notree")
    crl_lpm = as.data.frame(pro_pc)[crl_lpm_ind, ]

    sel = which(x_pc_id[,1] == 1)
    crl_lpm_idx <- sel[crl_lpm_ind]

    return(list(crl_lpm=crl_lpm, crl_lpm_idx=crl_lpm_idx))
}


do_psm = function(trt, big){
    x_all = rbind(trt, big)
    T = c(rep(1, nrow(trt)), rep(0, nrow(big)))
    m.out1 <- matchit(T ~ .,
                        data = x_all,
                        method = 'nearest',
                        distance = "glm")
    select_index = as.integer(m.out1$match.matrix[, 1])
    crl_psm <- big[select_index, ]
    cat("number of NAs in PSM: ", sum(is.na(crl_psm[, 1])), "\n")
    return(list(crl_psm=crl_psm, crl_psm_idx=select_index))
}

do_ebal = function(trt, excrl){
    treatment = c(rep(1, nrow(trt)), rep(0, nrow(excrl)))
    colnames(trt) = colnames(excrl)
    X = rbind(trt, excrl)
    # remove colinearity
    X_controls <- X[treatment == 0, ]
    lin_combos <- findLinearCombos(X_controls)
    print(lin_combos$remove)
    # Drop those columns from your FULL matrix X
    if (!is.null(lin_combos$remove)) {
      X_clean <- X[, -lin_combos$remove]
    } else {
      X_clean <- X
    }
    # add moment
    p <- ncol(X_clean)
    # target_cols <- (p-4):p # for parker
    target_cols <- (p-1):p # for mimic
    X_clean[, target_cols] <- scale(X_clean[, target_cols])
    X_clean <- cbind(X_clean, X_clean[, target_cols]^2)
    X_clean <- cbind(X_clean, X_clean[, target_cols]^3)

    eb.out <- ebalance(Treatment=treatment,
                      X=X_clean)
    return(eb.out$w)
}

In [ ]:
%%R

## Function to run in r
run_in_r = function(file_path,
                    w_opt, x_pc_id,
                    pro_trt, pro_big, pro_pc,
                    use_trt, use_big, use_pc,
                    sl_score, save_sl_results, rep_select, n_rep_select, psm, name, w_adjust){

    w_opt = as.vector(w_opt)
    x_pc_id = as.data.frame(x_pc_id)

    use_pc <- as.data.frame(use_pc)
    use_trt <- as.data.frame(use_trt)
    use_big <- as.data.frame(use_big)

    pro_trt = as.data.frame(pro_trt)
    pro_big = as.data.frame(pro_big)
    pro_pc = as.data.frame(pro_pc)

    if(sl_score){

        if(rep_select){

            crl_lpm_lst = list()
            crl_psm_lst = list()
            crl_lpm_idx_lst = list()
            crl_psm_idx_lst = list()
            sl_score_lpm_lst = list()
            sl_score_psm_lst = list()

            for (t in 1:n_rep_select){
                cat("LPM SL score...", "\n")
                res_lpm = do_lpm(w_opt, x_pc_id, use_trt, use_pc, pro_pc, w_adjust)
                crl_lpm_lst[[t]] = res_lpm$crl_lpm
                crl_lpm_idx_lst[[t]] = res_lpm$crl_lpm_idx
                sl_score_lpm_lst[[t]] = sl_score(pro_trt, res_lpm$crl_lpm)

                # if(psm){
                #     cat("PSM SL score...", "\n")
                #     res_psm = do_psm(pro_trt, pro_big)
                #     crl_psm_lst[[t]] = res_psm$crl_psm
                #     crl_psm_idx_lst[[t]] = res_psm$crl_psm_idx
                #     sl_score_psm_lst[[t]] = sl_score(pro_trt, na.omit(res_psm$crl_psm))
                # }


            }

            if(save_sl_results){
                cat("Save final controls selection results...", "\n")
                dir.create(file.path(file_path, name), recursive = TRUE, showWarnings = FALSE)

                saveRDS(crl_lpm_lst, file=file.path(file_path, name, "crl_lpm_lst.rds"))
                saveRDS(crl_lpm_idx_lst, file=file.path(file_path, name, "crl_lpm_idx_lst.rds"))
                saveRDS(sl_score_lpm_lst, file=file.path(file_path, name, "sl_score_lpm_lst.rds"))

                # saveRDS(crl_psm_lst, file=file.path(file_path, name, "crl_psm_lst.rds"))
                # saveRDS(crl_psm_idx_lst, file=file.path(file_path, name, "crl_psm_idx_lst.rds"))
                # saveRDS(sl_score_psm_lst, file=file.path(file_path, name, "sl_score_psm_lst.rds"))
            }

        }
        else{
            # Do LPM
            cat("Do LPM...", "\n")
            res_lpm = do_lpm(w_opt, x_pc_id, use_trt, use_pc, pro_pc)
            crl_lpm = res_lpm$crl_lpm
            sl_score_lpm = sl_score(pro_trt, crl_lpm)

            if(psm){
                # Do Propensity score matching
                cat("Do PSM...", "\n")
                res_psm = do_psm(pro_trt, pro_big)
                crl_psm = res_psm$crl_psm
                sl_score_psm = sl_score(pro_trt, na.omit(crl_psm))
            }

            if(save_sl_results){
                cat("Save final controls selection results...", "\n")
                dir.create(file.path(file_path, name), recursive = TRUE, showWarnings = FALSE)

                saveRDS(crl_lpm, file=file.path(file_path, name, "crl_lpm.rds"))
                saveRDS(res$crl_lpm_idx, file=file.path(file_path, name, "crl_lpm_idx.rds"))
                saveRDS(sl_score_lpm, file=file.path(file_path, name, "sl_score_lpm.rds"))

                saveRDS(crl_psm, file=file.path(file_path, name, "crl_psm.rds"))
                saveRDS(res$crl_psm_idx, file=file.path(file_path, name, "crl_psm_idx.rds"))
                saveRDS(sl_score_psm, file=file.path(file_path, name, "sl_score_psm.rds"))
            }

        }
    }

}





In [ ]:
%%R

# comparing EBAL
# Do EBAL on big, then random sampling
## Function to run in r
run_in_r = function(file_path,
                    w_opt, x_pc_id,
                    pro_trt, pro_big, pro_pc,
                    use_trt, use_big, use_pc,
                    sl_score, save_sl_results, rep_select, n_rep_select, psm, name, w_adjust){

    w_opt = as.vector(w_opt)
    x_pc_id = as.data.frame(x_pc_id)

    use_pc <- as.data.frame(use_pc)
    use_trt <- as.data.frame(use_trt)
    use_big <- as.data.frame(use_big)

    pro_trt = as.data.frame(pro_trt)
    pro_big = as.data.frame(pro_big)
    pro_pc = as.data.frame(pro_pc)

    if(sl_score){

        cat("Do EBAL on BIG...", "\n")
        w_ebal_big = do_ebal(pro_trt, pro_big)

        cat("Do EBAL on PC...", "\n")
        w_ebal_pc = do_ebal(pro_trt, pro_pc)

        if(rep_select){
            crl_ebalbig_lst = list()
            sl_score_ebalbig_lst = list()
            crl_ebalbig_idx_lst = list()

            crl_ebal_lst = list()
            sl_score_ebal_lst = list()
            crl_ebal_idx_lst = list()

            crl_ebal_incl_lst = list()
            sl_score_ebal_incl_lst = list()
            crl_ebal_incl_idx_lst = list()

            crl_psm_lst = list()
            sl_score_psm_lst = list()
            crl_psm_idx_lst = list()


            for (t in 1:n_rep_select){
                cat("PSM SL score...", "\n")
                res_psm = do_psm(pro_trt, pro_big)
                crl_psm_lst[[t]] = res_psm$crl_psm
                crl_psm_idx_lst[[t]] = res_psm$crl_psm_idx
                sl_score_psm_lst[[t]] = sl_score(pro_trt, na.omit(res_psm$crl_psm))

                cat("EBAL SL score on BIG...", "\n")
                sampled_control_indices <- sample(
                  x = 1:nrow(pro_big),
                  size = nrow(pro_trt),
                  replace = FALSE,
                  prob = w_ebal_big          # Your ebal weights go here
                )

                # Extract the actual rows
                crl_sampled <- pro_big[sampled_control_indices, ]
                crl_ebalbig_lst[[t]] = crl_sampled
                crl_ebalbig_idx_lst[[t]] = sampled_control_indices
                sl_score_ebalbig_lst[[t]] = sl_score(pro_trt, crl_sampled)


                cat("EBAL SL score on PC...", "\n")
                res_lpm = do_lpm(w_ebal_pc, x_pc_id, use_trt, use_pc, pro_pc, w_adjust="none")
                crl_ebal_lst[[t]] = res_lpm$crl_lpm
                crl_ebal_idx_lst[[t]] = res_lpm$crl_lpm_idx
                sl_score_ebal_lst[[t]] = sl_score(pro_trt, res_lpm$crl_lpm)

                cat("EBAL SL score on PC with incl...", "\n")
                res_lpm = do_lpm(w_ebal_pc, x_pc_id, use_trt, use_pc, pro_pc, w_adjust="incl")
                crl_ebal_incl_lst[[t]] = res_lpm$crl_lpm
                crl_ebal_incl_idx_lst[[t]] = res_lpm$crl_lpm_idx
                sl_score_ebal_incl_lst[[t]] = sl_score(pro_trt, res_lpm$crl_lpm)


            }

            if(save_sl_results){
                cat("Save final controls selection results...", "\n")
                dir.create(file.path(file_path, name), recursive = TRUE, showWarnings = FALSE)

                saveRDS(crl_psm_lst, file=file.path(file_path, name, "crl_psm_lst.rds"))
                saveRDS(crl_psm_idx_lst, file=file.path(file_path, name, "crl_psm_idx_lst.rds"))
                saveRDS(sl_score_psm_lst, file=file.path(file_path, name, "sl_score_psm_lst.rds"))

                saveRDS(crl_ebal_lst, file=file.path(file_path, name, "crl_ebal_lst.rds"))
                saveRDS(crl_ebal_idx_lst, file=file.path(file_path, name, "crl_ebal_idx_lst.rds"))
                saveRDS(sl_score_ebal_lst, file=file.path(file_path, name, "sl_score_ebal_lst.rds"))

                saveRDS(crl_ebal_incl_lst, file=file.path(file_path, name, "crl_ebal_incl_lst.rds"))
                saveRDS(crl_ebal_incl_idx_lst, file=file.path(file_path, name, "crl_ebal_incl_idx_lst.rds"))
                saveRDS(sl_score_ebal_incl_lst, file=file.path(file_path, name, "sl_score_ebal_incl_lst.rds"))

                saveRDS(crl_ebalbig_lst, file=file.path(file_path, name, "crl_ebalbig_lst.rds"))
                saveRDS(crl_ebalbig_idx_lst, file=file.path(file_path, name, "crl_ebalbig_idx_lst.rds"))
                saveRDS(sl_score_ebalbig_lst, file=file.path(file_path, name, "sl_score_ebalbig_lst.rds"))
            }

        }

    }

}


In [ ]:
run_in_r = globalenv['run_in_r']

def run_all(trt_path, big_path, file_path, configs, return_result_type,
            train_vae, adjust, save_vae_model, save_vae_results, save_vae_detail,
            train_kmmd, save_kmmd_results, save_kmmd_detail,
            sl_score, save_sl_results, rep_select, n_rep_select, psm, name, w_adjust):

    # Do before lpm

    dataset_trt = pyreadr.read_r(trt_path)
    dataset_trt = list(dataset_trt.items())[0][1]

    Bigdataset = pyreadr.read_r(big_path)
    Bigdataset = list(Bigdataset.items())[0][1]

    w_opt, x_pc_id, org_trt, org_big, org_pc, use_trt, use_big, use_pc = run_full_simulation_from_data(
        data=dataset_trt,
        big_data=Bigdataset,
        configs=configs,  # dictionary with training params per
        output_root=file_path,
        train_vae=train_vae,
        adjust=adjust,
        save_vae_model=save_vae_model,
        save_vae_results=save_vae_results,
        save_vae_detail=save_vae_detail,
        train_kmmd=train_kmmd,
        save_kmmd_results=save_kmmd_results,
        save_kmmd_detail = save_kmmd_detail,
        return_result_type=return_result_type,
        device=device
    )

    run_in_r(file_path,
             w_opt, x_pc_id,
             org_trt, org_big, org_pc,
             use_trt, use_big, use_pc,
             sl_score, save_sl_results, rep_select, n_rep_select, psm, name, w_adjust)


# Implementation demo for "scdp"

In [ ]:
configs = {
    'preprocess': {
        'binary_indices': [0, 1, 2, 3, 4],
        'categorical_indices_with_levels': {5:4, 6:6},
        'ordinal_indices': [7, 8],
        'continuous_indices': [9]
    },
    'vae':{
        'binary': {
            'batch_size': 128,
            'epochs': 1000,
            'hidden_dim': 50,
            'n_components': 2,
            'lr': 0.005,
            'scheduler': 'cosine'
        },
        'ordinal': {
            'batch_size': 128,
            'epochs': 1000,
            'hidden_dim': 50,
            'n_components': 2,
            'ordinal_K': 4,
            'embedding_dim': 4,
            'lr': 0.005,
            'scheduler': 'cosine'
        },
        'cont': {
            'batch_size': 128,
            'epochs': 1000,
            'hidden_dim': 50,
            'n_components': 1,
            'lr': 0.005,
            'scheduler': 'cosine'
        },
        'latent':{
            'batch_size': 128,
            'epochs': 1000,
            'hidden_dim': 32,
            'n_components': 3,
            'lr': 0.005,
            'scheduler': 'cosine'
        }
    },
    'kmmd':{
        'epochs':int(5e4),
        'lr':0.01,
        'ssl_penalty':False,
        'lambda0':5,
        'lambda1':0.1,
        'method': 'eucl',
        'use_latent': False,
        'adjust': False,
        'scheduler': 'cosine'
    },
    'detection':{
              'n_same':40000
    }

}

run_seed(seed=42)

In [ ]:
run_all(trt_path='scdp/trt.rds', big_path='scdp/big.rds',
        file_path='scdp',
        configs=configs, return_result_type='processed',
        train_vae=True, save_vae_model=True, save_vae_results=True, save_vae_detail=True,
        train_kmmd=True, save_kmmd_results=True, save_kmmd_detail=True,
        sl_score=True, save_sl_results=True, rep_select=True, n_rep_select=5, psm=True, name='', w_adjust="none")